# Silver Layer – Product Information

This notebook processes product data from the Bronze layer and prepares it for the Silver layer by applying data cleansing, validation, standardization, and transformation rules.

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.table('workspace.bronze.crm_prd_info') 


In [0]:
df = ( 
      df.withColumn('key' , regexp_replace(substring(col("prd_key") , 1,5) , '-' , '_' ) ) 
      .withColumn('cat' , substring(col("prd_key") ,7 , length(col("prd_key")) ) )
      )

## Handle Missing Values

Replace missing product cost values with 0.

In [0]:
df = df.fillna({'prd_cost' : 0})

## Standardize Product Line

Convert product line codes to descriptive values.

In [0]:
df = (
    df.withColumn('prd_line' ,
                  when(trim(col('prd_line') ) == 'M' , 'Mountain' )
                  .when(trim(col('prd_line') ) == 'R' , 'Road' )
                  .when(trim(col('prd_line') ) == 'S' , 'Other Sales' )
                  .when(trim(col('prd_line') ) == 'T' , 'Touring' )
                  .otherwise('N/A')
                  )
)

## Fix Product End Dates

Replace invalid end dates with the next product start date within each product key.

In [0]:
w =  Window.partitionBy('prd_key').orderBy(col('prd_start_dt'))
df =  df.withColumn('prd_end_dt' , lead('prd_start_dt').over(w) )

In [0]:
df = df.select(
    "prd_id",
    "prd_key",
    "key",
    "cat",
    "prd_nm",
    "prd_cost",
    "prd_line",
    "prd_start_dt",
    "prd_end_dt"
)


## Rename Columns

Rename columns to follow the Silver layer naming convention.

In [0]:
rename_map  = {
    'prd_id' : 'product_id',
    'key'  : 'product_number'  ,
    'cat' :  'category_id' , 
    'prd_nm' : 'product_name' ,
    'prd_cost' : 'product_cost' ,
    'prd_line' : 'product_line' ,
    'prd_start_dt' : 'start_date' ,
    'prd_end_dt' : 'end_date'

}

for old_name , new_name in rename_map.items() :
    df =  df.withColumnRenamed(old_name, new_name)
df = df.orderBy('product_id')


## Load to Silver Layer

Write the cleaned and transformed product data to the Silver layer.

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.crm_prd_info')

## Verify Data Load

Validate that the product data has been successfully loaded into the Silver layer.

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_prd_info LIMIT 10